# Local group-specific LLM: fine-tuning and evaluation

This notebook fine-tunes an open-weight model to request approved project tools and write evidence-bound **English** research summaries. It does not replace the deterministic finance pipeline: Python remains responsible for calculation, quality gates, point-in-time controls, backtests, and evidence commitment.

- **Mac / CPU:** dataset generation, review, validation, and optional inference smoke test.
- **Google Colab CUDA:** QLoRA training and held-out evaluation.
- **Academic rule:** do not train on cloud-LLM-generated answers. Every training example must be reviewed by the group.

## 1. Project location and optional dependencies

For Colab, first place this repository in the runtime (for example by cloning your own repository or uploading an archive), then set `PROJECT_ROOT` below. The model weights are downloaded from Hugging Face; no cloud text-generation API is used.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Change this in Colab if the repository is not the current directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # nbconvert runs from notebooks/
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Set PROJECT_ROOT to the repository folder that contains src/.')
os.chdir(PROJECT_ROOT)

# Run once per runtime. Core project dependencies are intentionally separate.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-llm.txt'])


In [ ]:
import json
import random

import torch

SEED = 481
random.seed(SEED)
torch.manual_seed(SEED)
HAS_CUDA = torch.cuda.is_available()
HAS_MPS = bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available())
DEVICE_PROFILE = 'cuda' if HAS_CUDA else 'mps' if HAS_MPS else 'cpu'
MODEL_NAME = os.getenv('BIST_LLM_MODEL', 'Qwen/Qwen3-4B' if HAS_CUDA else 'Qwen/Qwen3-0.6B')
RUN_TRAINING = HAS_CUDA  # Keep False on Mac/CPU. Colab users can change it after dataset review.
RUN_INFERENCE_SMOKE_TEST = False  # Downloads the selected model when enabled.
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'llm' / 'qwen_bist_lora'
DATASET_DIR = PROJECT_ROOT / 'data' / 'llm' / 'generated'

print({'device_profile': DEVICE_PROFILE, 'model': MODEL_NAME, 'run_training': RUN_TRAINING})
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Create and review training data

The generated rows are a transparent starter set from the fixed A-E catalog. They deliberately remain `human_reviewed: false`. Expand them with real deterministic scenario/evidence outputs, edit the English target summaries as a group, and mark only approved rows as reviewed before training. The splitter keeps each `case_id` in one split.

In [ ]:
from src.llm.training_data import (
    generate_seed_records, project_tool_definitions, validate_records, write_jsonl_splits,
)

seed_records = generate_seed_records()
paths = write_jsonl_splits(DATASET_DIR, seed_records)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))
print(json.dumps(project_tool_definitions(), indent=2))

# To train: replace this with group-reviewed records from JSONL, then mark each row true.
# Example only after a real review: record['metadata']['human_reviewed'] = True
try:
    validate_records(seed_records)
except ValueError as error:
    print('Expected pre-training stop:', error)


## 3. Select reviewed records

Set this to a reviewed JSONL export rather than the starter set for a submitted experiment. The explicit validation is intentional: it prevents accidentally training on unreviewed templates.

In [ ]:
REVIEWED_JSONL = PROJECT_ROOT / 'data' / 'llm' / 'reviewed_records.jsonl'

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

if REVIEWED_JSONL.exists():
    reviewed_records = read_jsonl(REVIEWED_JSONL)
    validate_records(reviewed_records, require_human_review=True)
    print(f'Using {len(reviewed_records)} reviewed records.')
else:
    reviewed_records = []
    print(f'No reviewed dataset at {REVIEWED_JSONL}. Dataset preparation is complete; training remains disabled.')

# TRIAL ONLY: pipeline smoke test on UNREVIEWED deterministic drafts + A-E seed records.
# Not valid for the submitted model; set False and use reviewed_records.jsonl for the real run.
ALLOW_UNREVIEWED_TRIAL = True
CANDIDATES_JSONL = PROJECT_ROOT / 'data' / 'llm' / 'candidates_v2' / 'candidates.jsonl'
if ALLOW_UNREVIEWED_TRIAL and not reviewed_records:
    reviewed_records = read_jsonl(CANDIDATES_JSONL) + seed_records
    for record in reviewed_records:
        record['metadata']['trial_unreviewed'] = True
    validate_records(reviewed_records, require_human_review=False)
    OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'llm' / 'qwen_bist_lora_TRIAL_unreviewed'
    print(f'TRIAL MODE: {len(reviewed_records)} unreviewed records -> {OUTPUT_DIR}')


## 4. QLoRA fine-tuning (CUDA only)

The base model is loaded in 4-bit NF4 only on CUDA. Mac/CPU users should leave `RUN_TRAINING=False`; they can still complete all data and validation steps above. Adapter weights, configuration, and trainer state are saved under `artifacts/llm/`, which is ignored by Git.

In [ ]:
if RUN_TRAINING:
    if not HAS_CUDA:
        raise RuntimeError('QLoRA training requires a CUDA runtime. Select a Colab GPU runtime or disable RUN_TRAINING.')
    if not reviewed_records:
        raise RuntimeError('Add group-reviewed records before training.')

    from datasets import Dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer

    from src.llm.training_data import stable_split

    splits = stable_split(reviewed_records)
    train_dataset = Dataset.from_list(splits['train']).remove_columns(['metadata'])
    validation_dataset = Dataset.from_list(splits['validation']).remove_columns(['metadata'])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization, device_map={'': 0})
    peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM')
    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        max_length=3072,  # evidence-JSON prompts are ~1.5-2k tokens; 1024 would cut off the answer
        gradient_checkpointing=True,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),  # T4 has no bf16
        report_to='none',
        seed=SEED,
    )
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / 'adapter'))
    print('Saved adapter to', OUTPUT_DIR / 'adapter')


## 5. Guarded tool routing and held-out evaluation

A model response is not executed directly. The router accepts only a single approved JSON tool request, rejects extra arguments such as `allow_download`, and pins every request to the original cutoff. A full analysis should still be run through `HarnessSession`, which enforces the complete state machine.

In [ ]:
from src.llm.tool_loop import SafeProjectToolRouter, ToolRequestError

router = SafeProjectToolRouter()
safe_request = json.dumps({
    'tool_name': 'check_data_quality',
    'arguments': {'ticker': 'ASELS', 'as_of_date': '2025-09-30'},
})
print('Approved tools:', router.allowed_tools)
print('Example model request:', safe_request)
# Execution requires the repository's local market cache:
# tool_result = router.execute_json(safe_request, as_of_date='2025-09-30')

def report_guardrail_metrics(texts):
    lowered = [text.lower() for text in texts]
    return {
        'response_count': len(texts),
        'direct_trade_language': sum(('buy' in text or 'sell' in text) for text in lowered),
        'guaranteed_return_language': sum(('guaranteed' in text or 'certain return' in text) for text in lowered),
        'evidence_reference_missing': sum(('e1' not in text and 'e2' not in text) for text in lowered),
    }

# For the final report, generate base-model and adapter answers for the same held-out
# report records, then compare these metrics alongside src.experiments A-E results.


## Submission checklist

Record the base model ID, adapter directory/hash, dependency versions, seed, dataset manifest, GPU type, training parameters, and held-out guardrail results. Include representative English summaries with their evidence IDs. Do not commit the dataset, checkpoint, or adapter unless your course policy explicitly permits it.